## Generative models for modality integration

In [ ]:
import os
import sys
import gc

import tifffile
import numpy as np
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from ml_collections import ConfigDict
from skimage import io
from skimage.transform import rescale
from skimage.exposure import rescale_intensity
from skimage.color import rgb2gray

In [ ]:
sys.path.append('..')
sys.path.append('../models/')

import utils, configs, constants
from dataset import IMSDataset
import bvae, model_train, model_eval, IO

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

### Load H&E images

In [ ]:
data_path = '../data/he_imgs/aligned/non_rigid/'

filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']
he_imgs = [tifffile.imread(os.path.join(data_path, f))[...,:3]
           for f in filenames]

In [ ]:
# Trim middle FOV
ylow, yhigh = 600, 3200
xlow, xhigh = 500, 3100
imgs_trimmed = []
for img in he_imgs:
    imgs_trimmed.append(
        np.array([chnl for chnl in img[ylow:yhigh, ylow:yhigh, :]]).transpose(2,0,1)
    )

del ylow, yhigh

In [ ]:
def disp_he_chans(imgs, title=None, ncols=4):
    """Display RGB tHistology images"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx].transpose(1,2,0))
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
disp_he_chans(imgs_trimmed)

In [ ]:
# Generate training samples

# Rescale to high-re (512 x 512) images
out_path = '../data/he_imgs/hires/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

in_dim = imgs_trimmed[0].shape[-1]
out_dim = 512
scale = out_dim / in_dim

for filename, img in zip(filenames, imgs_trimmed):
    filename = filename.split('.')[0] + '.tif'
    img_hires = rescale(img, scale=scale, preserve_range=True, channel_axis=0)
    img_hires /= 255.0
    tifffile.imwrite(os.path.join(out_path, filename), img_hires)

del filename, img, img_hires, out_path

# Rescale to low-res (128 x 128) images
# out_path = '../data/he_imgs/lowres/'
# if not os.path.exists(out_path):
#     os.makedirs(out_path, exist_ok=True)

# in_dim = imgs_trimmed[0].shape[-1]
# out_dim = 128
# scale = out_dim / in_dim

# for filename, img in zip(filenames, imgs_trimmed):
#     img_lowres = rescale(img, scale=scale, preserve_range=True, channel_axis=0)
#     img_lowres /= 255.0
#     tifffile.imwrite(os.path.join(out_path, filename), img_lowres)

# del filename, img, img_lowres, out_path

### $\beta$-VAE for H&E reconstruction

In [ ]:
from importlib import reload
reload(bvae)
reload(model_train)
reload(model_eval)

In [ ]:
model_configs = configs.set_model_configs(
    drop_rate = 0.1,
    beta = 2,
    ydim = 128,
    xdim = 128,
    batch_size = 1,
    lengthscale = 0.5,
    device = torch.device('cpu')
)

train_configs = configs.set_train_configs(
    data_path = '../data/he_imgs/lowres/',
    lr = 1e-4,
    n_epochs = 500
)

dataloader = DataLoader(
    IMSDataset(
        data_path=train_configs.data_path,
        norm_stats=(constants.HE_MEAN, constants.HE_STD)
    ),
    batch_size=model_configs.batch_size
)


In [ ]:
model = bvae.BetaVAE2D(model_configs)
model, loss_dict = model_train.train(model,
                                     dataloader,
                                     train_configs, 
                                     model_configs)

In [ ]:
loss_tot = torch.tensor(loss_dict['total']).cpu().numpy()
loss_nll = torch.tensor(loss_dict['NLL']).cpu().numpy()
loss_kl = torch.tensor(loss_dict['KL']).cpu().numpy()
loss_gp = torch.tensor(loss_dict['GP']).cpu().numpy()

In [ ]:
n_epochs = train_configs.n_epochs
beta = model_configs.beta

step = 10

plt.figure(figsize=(10, 4))
plt.plot(np.arange(n_epochs)[::step], 1e-1*loss_tot[::step], '.--',  label = '1e-1*Total loss')
plt.plot(np.arange(n_epochs)[::step], 1e-1*loss_nll[::step], '.--', color='orange', label = r'1e-1*$\Vert x - \hat{x} \Vert^2$')
plt.plot(np.arange(n_epochs)[::step], loss_kl[::step], '.--', color='green', label = r'$D_{\text{KL}}(q(z \mid x) || p(z))$')
plt.plot(np.arange(n_epochs)[::step], 1e3*loss_gp[::step], '.--', color='pink', label = r'1e3*GP loss')

plt.title('Training logs')
plt.legend()
plt.show()


In [ ]:
imgs, x_preds, latents, qz_preds = model_eval.predict(model, 
                                                      dataloader=dataloader,
                                                      device=model_configs.device, 
                                                      norm_stats=(constants.HE_MEAN, constants.HE_STD),
                                                      normalize_per_channel=False,
                                                      batch_size=1)

In [ ]:
for i in range(len(x_preds)):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, dpi=200)
    ax1.imshow(imgs[i].transpose(1,2,0))
    ax1.set_title('Input image')
    ax1.axis('off')
    
    ax2.imshow(x_preds[i].transpose(1,2,0))
    ax2.set_title('Reconstructed image')
    ax2.axis('off')

    ax3.imshow(qz_preds[i], cmap='coolwarm')
    ax3.set_title(r'$q(z|x)$')
    ax3.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
latents = [model_eval.get_latent(model, img, norm_stats=(constants.HE_MEAN, constants.HE_STD)).flatten() for img in imgs]

R1 = np.corrcoef(latents)
sns.heatmap(R1, vmax=0.75, cmap='coolwarm')
plt.title('Correlation (latent variables) across L')
plt.show()

del R1

### Load CyIF images
- Create low-dim noisy estimation on zonation dynamics $p(z)$ w/ determinsitic heat diffusion

In [ ]:
from skimage.exposure import equalize_adapthist, rescale_intensity
from skimage.transform import rescale
from skimage.filters import sobel, threshold_otsu, threshold_minimum
from skimage.filters import gaussian as gaussian_blur

sys.path.append('..')
import IO, zonation

In [ ]:
from importlib import reload
reload(IO)
reload(zonation)

In [ ]:
def disp_chans(imgs, title=None, ncols=4):
    """Display single-channel zonation image"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx])
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

def disp_cyif_chans(imgs, title=None, ncols=4):
    """Display 3-channel zonation image"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx].transpose(1,2,0))
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
data_path = '../data/cycif/aligned/non_rigid/'
cyif_annot_imgs = IO.load_annot_tiffs(data_path, ext='ome.tif')
filenames = list(cyif_annot_imgs.keys())

In [ ]:
# Select channels to keep
chans_to_keep = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68',
]

imgs = [np.array([img[label] for label in chans_to_keep])
        for _, img in cyif_annot_imgs.items()]

#### Generate training patches

In [ ]:
# Trim middle FOV
radius = 2048
imgs_trimmed = []
for img in imgs:
    ny, nx = img.shape[-2:]
    ylow = ny//2 - radius
    yhigh = ny//2 + radius
    imgs_trimmed.append(
        np.array([chnl for chnl in img[:, ylow:yhigh, ylow:yhigh]])
    )

del img, ny, nx, ylow, yhigh

In [ ]:
# Rescale to high-res (4096 x 4096) images
out_path = '../data/cycif/zonation_hires/'
if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

for filename, img in zip(filenames, imgs_trimmed):
    filename = filename.split('.')[0] + '.tif'
    tifffile.imwrite(os.path.join(out_path, filename), img)

del filename, img, out_path

**Generate Heat Diffusion priors**

In [ ]:
# Load training patches

data_path = '../data/cycif/zonation_hires/'
# data_path = '../data/cycif/zonation_lowres/'
filenames = [f for f in sorted(os.listdir(data_path))
             if f[-3:] == 'tif']

imgs = [tifffile.imread(os.path.join(data_path, f))
        for f in filenames]

# Compute channel-wise statistics
# chan_means = np.array([img.mean((1,2)) for img in imgs]).mean(0)
# chan_stds = np.array([img.std((1,2)) for img in imgs]).mean(0)

In [ ]:
def create_vein_prior(img, cv_chan_idx, pv_chan_idx, sigma=1.5):
    assert img.ndim == 3 , "Requires 3-dim zonation marker images"
    cv_chan = gaussian_blur(img[cv_chan_idx], sigma=sigma)
    cv_threshold = threshold_otsu(cv_chan)
    cv_mask = (cv_chan > cv_threshold).astype(np.uint8)

    pv_chan = gaussian_blur(img[pv_chan_idx])
    pv_threshold = threshold_otsu(pv_chan)
    pv_mask = (pv_chan > pv_threshold).astype(np.uint8)

    vein_prior = np.zeros_like(cv_chan, dtype=np.int8)
    vein_prior[np.logical_and(pv_mask == 1, cv_mask == 0)] = -1
    vein_prior[np.logical_and(pv_mask == 0, cv_mask == 1)] = 1

    return vein_prior
    

**DEBUG: equal sampling on CV/PV priors or graph construction w/ nuclei locations** 

In [ ]:
vein_path = '../data/cycif/zonation_mask/'
# if not os.path.exists(vein_path):
#     os.makedirs(vein_path, exist_ok=True)


vein_priors = []
for filename, img in zip(filenames, imgs):
    vein_prior = create_vein_prior(img, cv_chan_idx=1, pv_chan_idx=3)
    vein_priors.append(vein_prior)

#     filename = 'CyIF_vein_prior_{}.tif'.format(filename.split('.')[0][-2:])
#     tifffile.imwrite(os.path.join(vein_path, filename), vein_prior)

# del filename, img, vein_prior
    
# # Load from file
# vein_priors = np.array([tifffile.imread(os.path.join(vein_path, f))
#                         for f in sorted(os.listdir(vein_path))
#                         if 'vein_prior' in f])

In [ ]:
# Sample equal # CV / PV boundary pts
def sample_priors(prior, is_3d=False, n=2000):
    cv_coords = np.where(prior == 1)
    indices = np.random.choice(np.arange(len(cv_coords[0])), size=n, replace=False)
    if is_3d:
        sampled_cv_coords = tuple([cv_coords[0][indices], 
                                   cv_coords[1][indices],
                                   cv_coords[2][indices]])
    else:
        sampled_cv_coords = tuple([cv_coords[0][indices], 
                                   cv_coords[1][indices]])

    pv_coords = np.where(prior == -1)
    indices = np.random.choice(np.arange(len(pv_coords[0])), size=n, replace=False)
    if is_3d:
        sampled_pv_coords = tuple([pv_coords[0][indices], 
                                   pv_coords[1][indices],
                                   pv_coords[2][indices]])
    else:
        sampled_pv_coords = tuple([pv_coords[0][indices], 
                                   pv_coords[1][indices]])

    masked_prior = np.zeros_like(prior)
    masked_prior[sampled_cv_coords] = 1
    masked_prior[sampled_pv_coords] = -1

    return masked_prior

masked_vein_priors = np.array([sample_priors(img)
                               for img in vein_priors])

In [ ]:
masked_vein_priors = rescale(masked_vein_priors, 0.125, preserve_range=True, channel_axis=0)

In [ ]:
# 2D heat diffusion
# Us_2d = []  # Heat along axial-direction
# for vein_prior in vein_priors:
#     model = zonation.HeatDiffusion(vein_prior=vein_prior, ndim=2)
#     U_i, interior_nodes = model.get_interior_U()
#     U = model.infer_zone_dynamics()
#     Us_2d.append(U)


# 3D heat diffusion 
model = zonation.HeatDiffusion(vein_prior=np.array(masked_vein_priors), ndim=3, anis=20)
U_i, interior_nodes = model.get_interior_U()
Us = model.infer_zone_dynamics()
lobule_layers = model.infer_zones()


In [ ]:
def disp_zone_temps(imgs, title=None, ncols=4, cmap='coolwarm'):
    """Display 3-channel zonation image"""
    depth = len(imgs)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(imgs[idx], cmap=cmap)
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
disp_zone_temps(lobule_layers, title='Zonations', cmap='plasma')

In [ ]:
filenames = sorted(os.listdir('../data/cycif/zonation_mask'))
Us = [tifffile.imread(os.path.join('../data/cycif/zonation_mask', f)) for f in filenames]

In [ ]:
# Save zonation priors p(z)

vein_path = '../data/cycif/zonation_mask/'
if not os.path.exists(vein_path):
    os.makedirs(vein_path, exist_ok=True)

in_dim = 128
out_dim = 64
scale= out_dim / in_dim
pz_means = []

for filename, pz_mean in zip(filenames, Us):
    pz_mean_rescaled = rescale(pz_mean, scale=scale, preserve_range=True)
    filename = 'CyIF_dynamics_{}.tif'.format(filename.split('.')[0][-2:])
    tifffile.imwrite(os.path.join(vein_path, filename), pz_mean_rescaled)

del filename, pz_mean, pz_mean_rescaled

### Whether multi-channel images provide us additional inforamtion for dim-reduction?

In [ ]:
# Correlations + PCA
cyif_chans = [
    'DAPI',
    'GS 647', 
    'CYP3A4', 
    'ASS1 PE',
    'Pan CK',
    'CD31', 
    'CD45', 
    'CD56', 
    'CD68'
]

chan_corrs = [np.corrcoef(img.reshape(img.shape[0], -1))
              for img in imgs]


In [ ]:
def disp_corr_features(corrs, labels=None, titles=None, ncols=4):
    n_slices = len(corrs)
    nrows = n_slices // ncols if n_slices % ncols == 0 else n_slices // ncols + 1

    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 3*nrows), dpi=200)
    for r in range(nrows):
        for c in range(ncols):
            if idx >= n_slices:
                axes[r, c].axis('off')
                continue
            sns.heatmap(corrs[idx], mask=np.triu(corrs[idx]),
                        xticklabels=labels, yticklabels=labels,
                        vmin=-0.3, vmax=0.3, square=True, 
                        ax=axes[r, c], cmap='coolwarm')
            
            if titles is not None:
                axes[r, c].set_title(titles[idx])
            idx += 1
    plt.tight_layout()
    plt.suptitle('Feature correlations (Raw image)', fontsize=30, y=1.02)
    plt.show()
    return None
            

In [ ]:
disp_corr_features(chan_corrs, labels=cyif_chans, titles=filenames, ncols=6)

In [ ]:
fig, ax = plt.subplots(dpi=100)
sns.heatmap(np.mean(chan_corrs[5:15], axis=0), cmap='coolwarm', mask=np.triu(np.mean(chan_corrs, axis=0)),
            xticklabels=chans_to_keep, yticklabels=chans_to_keep,
            vmin=-0.3, vmax=0.3, ax=ax)
ax.set_title('Channel-wise correlations')
plt.show()

Slices w/ empty immune channels:<br>
`L = 0, 1, 2, 16, 18` (whether these are low-quality ones?

In [ ]:
for i, corr in enumerate(chan_corrs):
    fig, ax = plt.subplots(dpi=100)
    sns.heatmap(corr, cmap='coolwarm', mask=np.triu(corr),
                xticklabels=chans_to_keep, yticklabels=chans_to_keep,
                vmin=-0.3, vmax=0.3, ax=ax)
    ax.set_title('Channel-wise correlations (L={})'.format(i))
    plt.show()

### $\beta$-VAE for Cy-IF reconstruction

- Benchmark performance w/ linear layers vs. Conv2d
- Uninformative priors $p(z)$ (trying full covariance other than diag.)

In [ ]:
from importlib import reload

reload(utils)
reload(configs)
reload(bvae)
reload(model_train)
reload(model_eval)

#### Training b-VAE 1D

In [ ]:
model_configs = configs.set_model_configs(
    drop_rate = 0.2,
    beta = 0.33,
    pz_std = 1,
    c_base = 128,
    layer_mults = [8, 4],
    ydim = 128, 
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

train_configs = configs.set_train_configs(
    data_path = '../data/cycif/zonation_lowres/',
    lr = 1e-4,
    n_epochs = 100
)

dataloader = DataLoader(
    IMSDataset(
        data_path=train_configs.data_path,
        norm_stats=(constants.CYIF_MEAN, constants.CYIF_STD)
    ),
    batch_size=model_configs.batch_size
)


In [ ]:
model = bvae.BetaVAE(model_configs)

model, loss_dict = model_train.train(model,
                                     dataloader,
                                     train_configs, 
                                     model_configs)

In [ ]:
loss_tot = torch.tensor(loss_dict['total']).cpu().numpy()
loss_nll = torch.tensor(loss_dict['NLL']).cpu().numpy()
loss_kl = torch.tensor(loss_dict['KL']).cpu().numpy()

In [ ]:
n_epochs = train_configs.n_epochs
beta = model_configs.beta

step = 10

plt.figure(figsize=(6, 4))
plt.plot(np.arange(n_epochs)[::step], loss_tot[::step], '.--',  label = 'Total loss')
plt.plot(np.arange(n_epochs)[::step], loss_nll[::step], '.--', color='orange', label = r'$\Vert x - \hat{x} \Vert^2$')
plt.plot(np.arange(n_epochs)[::step], loss_kl[::step], '.--', color='green', label = r'$D_{\text{KL}}(q(z \mid x) || p(z))$')

plt.title('Training logs')
plt.legend()
plt.show()

In [ ]:
torch.save(model.state_dict(), 'results/bvae_1d_lowres_std_norm.pt')

#### Training b-VAE 2D

- **Delete L indices 0, 1, 2, 16, 18**

In [ ]:
# Model configs
model_configs = configs.set_model_configs(
    c_in = 8,
    c_out = 8,
    drop_rate = 0.1,
    beta = 0.5,
    pz_std = 1,
    ydim = 512,
    xdim = 512,
    latent_dim = 64,
    device = torch.device('cpu')
)

train_configs = configs.set_train_configs(
    data_path = '../data/cycif/zonation_hires/',
    prior_path = '../data/cycif/zonation_mask/',
    lr = 1e-4,
    n_epochs = 100
)

dataloader = DataLoader(
    IMSDataset(
        data_path=train_configs.data_path,
        prior_path=train_configs.prior_path,
        norm_stats=(constants.CYIF_MEAN, constants.CYIF_STD)
    ),
    batch_size=model_configs.batch_size
)

In [ ]:
model = bvae.BetaVAE2D(model_configs)

model, loss_dict = model_train.train(model,
                                     dataloader,
                                     train_configs, 
                                     model_configs)

In [ ]:
loss_tot = torch.tensor(loss_dict['total']).cpu().numpy()
loss_nll = torch.tensor(loss_dict['NLL']).cpu().numpy()
loss_kl = torch.tensor(loss_dict['KL']).cpu().numpy()

In [ ]:
n_epochs = train_configs.n_epochs
beta = model_configs.beta

step = 10

plt.figure(figsize=(6, 4))
plt.plot(np.arange(n_epochs)[::step], loss_tot[::step], '.--',  label = 'Total loss')
plt.plot(np.arange(n_epochs)[::step], loss_nll[::step], '.--', color='orange', label = r'$\Vert x - \hat{x} \Vert^2$')
plt.plot(np.arange(n_epochs)[::step], loss_kl[::step], '.--', color='green', label = r'$D_{\text{KL}}(q(z \mid x) || p(z))$')

plt.title('Training logs')
plt.legend()
plt.show()


In [ ]:
torch.save(model.state_dict(), 'results/bvae_2d_lowres_std_norm.pt')

#### Benchmark (held-out log-likelihood)

In [ ]:
model_1d_configs = configs.set_model_configs(
    drop_rate = 0.2,
    beta = 0.33,
    pz_std = 1,
    c_base = 128,
    layer_mults = [8, 4],
    ydim = 128, 
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

model_1d = bvae.BetaVAE(model_1d_configs)
model_1d.load_state_dict(torch.load('results/bvae_1d_lowres_std_norm.pt'))

model_2d_configs = configs.set_model_configs(
    drop_rate = 0.2,
    beta = 0.33,
    pz_std = 1,
    ydim = 128,
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

model_2d = bvae.BetaVAE2D(model_2d_configs)
model_2d.load_state_dict(torch.load('results/bvae_2d_lowres_std_norm.pt'))

dataset = CyIFDataset(data_path='../data/cycif/zonation_lowres_test/')
dataloader = DataLoader(dataset, batch_size=model_1d_configs.batch_size, shuffle=False)

In [ ]:
imgs, x_preds_1d, _ = model_eval.predict(model_1d, 
                                         dataloader = dataloader,
                                         device = model_1d_configs.device, 
                                         batch_size = 1)

_, x_preds_2d, _ = model_eval.predict(model_2d,
                                      dataloader = dataloader,
                                      device = model_2d_configs.device,
                                      batch_size = 1)

In [ ]:
MSEs_1d = [((x_true - x_pred)**2).mean()
           for (x_true, x_pred) in zip(imgs, x_preds_1d)]
MSEs_2d = [((x_true - x_pred)**2).mean()
           for (x_true, x_pred) in zip(imgs, x_preds_2d)]

plot_df = pd.DataFrame({
    'MSE': MSEs_1d + MSEs_2d,
    'Label': ['VAE (FC)']*len(imgs) + ['VAE (Conv2D)']*len(imgs)
})

sns.boxplot(x='Label', y='MSE', data=plot_df, hue='Label')
plt.title('Held-out avg. per-pixel MSE')
plt.show()

#### Predictions & Assessment

In [ ]:
model_1d_configs = configs.set_model_configs(
    drop_rate = 0.1,
    beta = 1,
    pz_std = 1,
    c_base = 128,
    layer_mults = [8, 4],
    ydim = 128, 
    xdim = 128,
    latent_dim = 256,
    device = torch.device('cpu')
)

model_2d_configs = configs.set_model_configs(
    drop_rate = 0.1,
    beta = 1,
    ydim = 128,
    xdim = 128,
    batch_size=1,
    device = torch.device('cpu')
)

In [ ]:
dataset = IMSDataset(
    data_path='../data/cycif/zonation_lowres/',
    norm_stats=(constants.CYIF_MEAN, constants.CYIF_STD)
)

do_2d = True
if do_2d:
    model_configs = model_2d_configs
    model = bvae.BetaVAE2D(model_configs)
    dataloader = DataLoader(dataset, batch_size=model_configs.batch_size, shuffle=False)
    model.load_state_dict(torch.load('results/bvae_2d_lowres_std_norm.pt'))
else:
    model_configs = model_1d_configs
    model = bvae.BetaVAE(model_configs)
    dataloader = DataLoader(dataset, batch_size=model_configs.batch_size, shuffle=False)
    model.load_state_dict(torch.load('results/bvae_1d_lowres_std_norm.pt'))

In [ ]:
imgs, x_preds, latents, qz_preds = model_eval.predict(model, 
                                                      dataloader=dataloader,
                                                      device=model_configs.device, 
                                                      batch_size=1)

### 3D integration of DESI (pos & neg ions) + Post DESI modalities

---